In [1]:
!pip freeze | grep raizen

# vc deve estar na raizenlib @ git+https://github.com/raizen-analytics/raizenlib.git@8fbe67dcceba4cd1504e659b398cc6b33718160b
# ou mais recente v3.6.4

# para atualizar: https://github.com/renatocastellani/ambiente_dev_raizen/blob/main/scripts/instal_raizenlib.sh

raizenlib @ file:///tmp/tmp.ugeGlVdGqd/raizenlib


In [2]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)             # Permite auto-ajuste da largura
pd.set_option('display.max_colwidth', None)      # Remove limite de largura de conteúdo


# Imports Gerais
from IPython.core.display import HTML

display(
    HTML("<style>pre { white-space: pre !important; }</style>")
)  # Para melhorar a visualização de DataFrames

# Vamos iniciar uma sessão do Spark similar ao que faríamos no Airflow
from raizenlib import framework
import pyspark.sql.functions as F
import pyspark.sql.types as T

#################################################################
# Caso precise de mais recursos, você pode aumentar o request_cpu
# e request_memory aqui de acordo com o selecionado na aba "Resources" do
# controlador de ambiente DEV.

executor_config = {
    "request_cpu": 0.2,
    "request_memory": "1Gi",
}

#################################################################

spark_conf = {}

spark = framework.spark.get_spark_session_ti(
    task_instance={
        "executor_config": executor_config
    },  # No airflow, isso seria o context['ti'] (é automático, então não precisamos nos preocupar)
    airflow_conn_id="SUA_SPN_CONN_ID",
    **spark_conf
)

# Em produção, o código acima seria substituído por algo como:
# spark = framework.spark.get_spark_session_ti(airflow_conn_id="SUA_SPN_CONN_ID")

print("-" * 30)
print("Spark Version:", spark.version)
print("Spark Threads:", spark.sparkContext.defaultParallelism)
print(
    "Spark Shuffle Partitions:", spark.conf.get("spark.sql.shuffle.partitions")
)


# from raizenlib.toolbox.sparktools import SparkTools
from raizenlib.utils.gen2_utils import Gen2Tools

gen2_tools = Gen2Tools()
# spark_tools = SparkTools()

[2026-03-03T18:40:38.858+0000] {__init__.py:18} ERROR - Failed to import framework.operators
[2026-03-03T18:40:39.053+0000] {sparktools.py:86} INFO - Cached credentials found!


26/03/03 18:40:40 WARN Utils: Your hostname, N-8HKNGH3 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/03 18:40:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 18:40:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


------------------------------
Spark Version: 3.5.3
Spark Threads: 1
Spark Shuffle Partitions: 2
[2026-03-03T18:40:42.725+0000] {gen2_utils.py:166} WARNING - MEMCACHE_PORT environment variable is not set.
[2026-03-03T18:40:42.853+0000] {gen2_utils.py:229} INFO - Trying to get cached credentials...
[2026-03-03T18:40:42.856+0000] {gen2_utils.py:233} INFO - Found cached credentials!
[2026-03-03T18:40:42.857+0000] {gen2_utils.py:236} INFO - Found cached authentication record!
[2026-03-03T18:40:43.269+0000] {gen2_utils.py:258} INFO - Cached credentials are still valid.
[2026-03-03T18:40:43.270+0000] {gen2_utils.py:264} INFO - Cached credentials loaded.


In [3]:

path_t001=gen2_tools.adls2_full_url('STAGING/SYSTEM/SAP_COMBS/FI/T001/ATUAL/')
df_t001 = spark.read.format("delta").load(path_t001)

In [4]:
df_t001.show(5,False)

26/03/03 18:41:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+----------+-----------------------+--------+-----------------+-------+-------+-----+-----+-------------------------+--------------+-----+-----+-----+-----+-----+-----+-----+-----+----------+-----+-----+-----+-----+-----+----------+----------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+--------+--------+-----+-------+-----+-----+--------+-------+----+--------+-----+------+------+------+------+------+-----+--------+-----------+-------------+
|AEKEY   |AEEPOCH   |AEDATTM                |AEOPFLAG|AEFILENAME       |AERUNID|AERECNO|MANDT|BUKRS|BUTXT                    |ORT01         |LAND1|WAERS|SPRAS|KTOPL|WAABW|PERIV|KOKFI|RCOMP|ADRNR     |STCEG|FIKRS|XFMCO|XFMCB|XFMCA|TXJCD     |FMHRDATE  |BUVAR|FDBUK|XFDIS|XVALV|XSKFN|KKBER|XMWSN|MREGL|XGSBE|XGJRV|XKDFT|XPROD|XEINK|XJVAA|XVVWA|XSLTA|XFDMM|XFD